# DeepLabV3 ResNet50 K-Fold Ultrasound Segmentation Pipeline
This notebook provides a complete interactive workflow to train, validate, and run inference using a DeepLabV3 ResNet50 architecture with patient-aware K-Fold cross validation.

**Objective**: Segment fetal head regions, extract outer boundary contours, and calculate perimeter (circumference) in pixels and physical units (mm/cm).

## 1. Setup Environment & Imports
Mount Google Drive and install requirements.

In [ ]:
# 1. Un-comment to Mount Google Drive (if using Colab)
# from google.colab import drive
# drive.mount('/content/drive')

# 2. Change path to your specific Google Drive project directory
%cd /content/drive/MyDrive/Dataset/Task - Segmentation/hypothesis2

# 3. Install required dependencies
!pip install -r ../requirements.txt

/content/drive/MyDrive/Dataset/Task - Segmentation/hypothesis2


In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("PyTorch version:", torch.__version__)

if os.path.exists("../images") and os.path.exists("../masks"):
    print(f"Found {len(os.listdir('../images'))} scans and {len(os.listdir('../masks'))} masks in parent folder.")
else:
    print("WARNING: Verify images and masks exist in the parent directory.")

Using device: cuda
PyTorch version: 2.11.0+cu128
Found 622 scans and 623 masks in parent folder.


## 2. Patient-Aware K-Fold Split Visualization

In [ ]:
from dataset import get_fold_split

train_imgs, train_masks, val_imgs, val_masks = get_fold_split(
    images_dir="../images",
    masks_dir="../masks",
    fold_idx=0,
    num_folds=5,
    random_seed=42
)

--- Fold 1/5 ---
Train patients: 400 | Images: 491
Val patients: 101 | Images: 131


## 3. Train the K-Fold DeepLabV3-ResNet50 Pipeline

In [ ]:
!python train.py \
    --config "config.json" \
    --images_dir "../images" \
    --masks_dir "../masks" \
    --epochs 15 \
    --batch_size 8 \
    --num_folds 5

Loading configurations from config file: config.json
Using device: cuda
DeepLabV3 parameters loaded. Pretrained: True

==================== TRAINING FOLD 1/5 ====================
--- Fold 1/5 ---
Train patients: 400 | Images: 491
Val patients: 101 | Images: 131
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth
100% 161M/161M [00:01<00:00, 116MB/s]  
/content/drive/MyDrive/Dataset/Task - Segmentation/hypothesis2/train.py:159: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
/content/drive/MyDrive/Dataset/Tas

## 4. Single Image Inference & Circumference Measurement

In [ ]:
from inference import load_model, run_inference, visualize_results

model_checkpoint = "checkpoints/fold_0_best.pth"
test_image_path = "../images/001_HC.png"

if os.path.exists(model_checkpoint):
    model = load_model(model_checkpoint, device)
    results = run_inference(
        model=model,
        image_path=test_image_path,
        img_size=(256, 256),
        threshold=0.5,
        pixel_spacing=0.1,
        device=device
    )
    print(f"--- Calculations for {os.path.basename(test_image_path)} ---")
    print(f"Circumference (Pixels): {results['perimeter_pixels']:.2f} px")
    print(f"Circumference (Physical): {results['perimeter_mm']:.2f} mm ({results['perimeter_mm']/10.0:.2f} cm)")
    visualize_results(results, test_image_path, save_path="outputs/inference_output.png")
else:
    print(f"Checkpoint {model_checkpoint} not found. Please train the model first.")

## 5. Run Batch Inference

In [ ]:
!python inference.py \
    --images_dir "../images" \
    --model_path "checkpoints/fold_0_best.pth" \
    --save_batch_dir "outputs/batch_inference"